# Reproducing the ZOD-Derived MFL Illustrative Case Study

This notebook accompanies the manuscript:

**Contradiction-Aware Mediative Fuzzy Logic: Operators, Semantic Coherence, Type-2/Type-3 Extensions, and Quantum Semantics**

**Author:** Oscar Montiel Ross

**Year:** 2026

## License

The code in this notebook is released under the MIT License.

Copyright (c) 2026 Oscar Montiel Ross

See the `LICENSE` file in this repository for the full license text.

## Data Notice

This notebook uses only a minimal preprocessed representation of selected ZOD-derived rows needed to reproduce the illustrative calculations reported in the manuscript.

No raw ZOD sensor data, images, LiDAR, radar, video, or personally identifying information are included in this repository.

The original Zenseact Open Dataset (ZOD) remains governed by its own license terms. Users should consult and cite the original ZOD dataset when using or referencing ZOD-derived material.

## Purpose of this Notebook

This notebook reproduces the deterministic numerical calculations reported in the ZOD-derived safety-first Mediative Fuzzy Logic (MFL) illustrative case study.

Starting from `zod_selected_preprocessed_rows.csv`, it recomputes:

* the contextual uncertainty score $c$;
* the proximity quantities $D$ and $r$;
* the mediative truth--falsity pair $(\mu_p,\nu_p)$;
* hesitation $\pi_p$;
* contradiction $\zeta_p$;
* the mediative score $M(\mu_p,\nu_p)$;
* the final safety-first decision category.

The computation is illustrative and reproducibility-oriented. It does not train, validate, or benchmark an autonomous-driving perception model.


This notebook reproduces the deterministic numerical calculations reported in
the ZOD-derived safety-first Mediative Fuzzy Logic (MFL) illustrative case study.

Starting from `zod_selected_preprocessed_rows.csv`, it recomputes:

- the contextual uncertainty score $c$;
- the proximity quantities $D$ and $r$;
- the mediative truth--falsity pair $(\mu_p,\nu_p)$;
- hesitation $\pi_p$;
- contradiction $\zeta_p$;
- the mediative score $M(\mu_p,\nu_p)$;
- the final safety-first decision category.

The computation is illustrative and reproducibility-oriented. It does not train,
validate, or benchmark an autonomous-driving perception model.

## Files

Place this notebook in the same folder as:

- `zod_selected_preprocessed_rows.csv`

Running all cells creates:

- `zod_mfl_recomputed_results.csv`

## How to run this notebook

Run the notebook from top to bottom using **Kernel → Restart & Run All**.

The calculation cells depend on variables created in earlier cells. If a later cell is
run before the recomputation cell, Jupyter will not yet know objects such as
`recomputed_rows`. The guarded cells in this version now provide a clear message
instead of a raw `NameError`.

## Coding rules

The contextual uncertainty score is

$$
c =
0.35c_{\mathrm{weather}}
+0.25c_{\mathrm{light}}
+0.20c_{\mathrm{road}}
+0.20c_{\mathrm{density}}.
$$

For positive-risk rows, the reported distance $d$ is converted into

$$
D=\operatorname{clip}(d/10,0,1),\qquad r=1-D.
$$

The truth--falsity pair is then

$$
\mu_p = 0.55+0.35r+0.10c,
\qquad
\nu_p = 0.05+0.20D+0.25c.
$$

For non-risk rows,

$$
\mu_p = 0.10+0.30c,
\qquad
\nu_p = 0.60+0.25(1-c).
$$

## Note on the verification table

The verification table compares two things:

- **computed**: the value recomputed by this notebook;
- **expected**: the value reported in the manuscript or stored in the input CSV;
- **ok**: whether the two values agree within the verification tolerance.

A small mismatch can occur when a value lies exactly on a decimal rounding boundary.
For example, in Case 2,

$$
\mu_p = 0.10 + 0.30(0.815) = 0.3445.
$$

The manuscript reports this value as $0.345$. Some Python formatting routines may
display the same boundary value as $0.344$, depending on binary floating-point
representation and rounding convention. This is a rounding/display issue, not a
mathematical discrepancy in the MFL computation.

To avoid false failures caused only by three-decimal display rounding, this notebook
uses a slightly more robust verification tolerance.

## Mediative quantities

Hesitation and contradiction are

$$
\pi_p=\max\{0,1-\mu_p-\nu_p\},
\qquad
\zeta_p=\max\{0,\mu_p+\nu_p-1\}.
$$

The mediative score is

$$
M(\mu_p,\nu_p)
=
\left(1-\pi_p-\frac{\zeta_p}{2}\right)\mu_p
+
\left(\pi_p+\frac{\zeta_p}{2}\right)(1-\nu_p).
$$

The safety-first decision rule is:

- $M(\mu_p,\nu_p)\ge 0.7$: decisive braking;
- $0.5\le M(\mu_p,\nu_p)<0.7$: cautious deceleration and additional sensing;
- $M(\mu_p,\nu_p)<0.5$: cautious monitoring.

In [1]:
from pathlib import Path
import csv
import math

INPUT_FILE = Path("zod_selected_preprocessed_rows.csv")
OUTPUT_FILE = Path("zod_mfl_recomputed_results.csv")

print(f"Input file:  {INPUT_FILE.resolve()}")
print(f"Output file: {OUTPUT_FILE.resolve()}")

Input file:  /Users/oross/Library/CloudStorage/Dropbox/2026/Articulos_Caps_sometidos_2026/Contradiction_Aware_FSS_2026/Suplementary/1st_Submission/MFL_ZOD_Supplementary_Material/zod_selected_preprocessed_rows.csv
Output file: /Users/oross/Library/CloudStorage/Dropbox/2026/Articulos_Caps_sometidos_2026/Contradiction_Aware_FSS_2026/Suplementary/1st_Submission/MFL_ZOD_Supplementary_Material/zod_mfl_recomputed_results.csv


In [2]:
def clip(value, lower=0.0, upper=1.0):
    """Clip value to the interval [lower, upper]."""
    return max(lower, min(upper, value))


def parse_float(value):
    """Parse a CSV field as float; return None for empty fields."""
    value = (value or "").strip()
    if value == "":
        return None
    return float(value)


def round3(value):
    """Return a three-decimal string, preserving empty values."""
    if value is None:
        return ""
    return f"{value:.3f}"

In [3]:
def compute_context_score(row):
    """
    Compute c = 0.35*c_weather + 0.25*c_light
                + 0.20*c_road + 0.20*c_density.

    The four component scores are read from the selected preprocessed rows.
    """
    return (
        0.35 * float(row["c_weather"])
        + 0.25 * float(row["c_light"])
        + 0.20 * float(row["c_road"])
        + 0.20 * float(row["c_density"])
    )


def compute_truth_falsity(row, c):
    """
    Compute D, r, mu_p, nu_p, pi_p, zeta_p, and M(mu_p,nu_p).

    Positive-risk rows use the distance field. Negative distances are clipped
    to D=0, corresponding to maximal proximity support in this illustration.
    """
    risk_indicator = int(row["risk_indicator"])
    distance_m = parse_float(row.get("distance_m", ""))

    if risk_indicator == 1:
        if distance_m is None:
            raise ValueError("Positive-risk rows require distance_m.")

        D = clip(distance_m / 10.0, 0.0, 1.0)
        r = 1.0 - D

        mu_p = 0.55 + 0.35 * r + 0.10 * c
        nu_p = 0.05 + 0.20 * D + 0.25 * c
    else:
        D = None
        r = None

        mu_p = 0.10 + 0.30 * c
        nu_p = 0.60 + 0.25 * (1.0 - c)

    pi_p = max(0.0, 1.0 - mu_p - nu_p)
    zeta_p = max(0.0, mu_p + nu_p - 1.0)

    M = (
        (1.0 - pi_p - zeta_p / 2.0) * mu_p
        + (pi_p + zeta_p / 2.0) * (1.0 - nu_p)
    )

    return {
        "D": D,
        "r": r,
        "mu_p": mu_p,
        "nu_p": nu_p,
        "pi_p": pi_p,
        "zeta_p": zeta_p,
        "M": M,
    }


def decision_from_score(M):
    """Apply the safety-first threshold rule."""
    if M >= 0.7:
        return "decisive braking"
    if M >= 0.5:
        return "cautious deceleration and additional sensing"
    return "cautious monitoring"

In [4]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Cannot find zod_selected_preprocessed_rows.csv. "
        "Run this notebook in the supplementary-material folder."
    )

recomputed_rows = []

with INPUT_FILE.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        c = compute_context_score(row)
        values = compute_truth_falsity(row, c)
        decision = decision_from_score(values["M"])

        recomputed_rows.append({
            "case_id": row["case_id"],
            "split": row["split"],
            "frame_id": row["frame_id"],
            "risk_indicator": row["risk_indicator"],
            "reason_field": row["reason_field"],
            "weather": row["weather"],
            "time_of_day": row["time_of_day"],
            "road_condition": row["road_condition"],
            "context_score_c": round3(c),
            "D": round3(values["D"]),
            "r": round3(values["r"]),
            "mu_p": round3(values["mu_p"]),
            "nu_p": round3(values["nu_p"]),
            "pi_p": round3(values["pi_p"]),
            "zeta_p": round3(values["zeta_p"]),
            "M": round3(values["M"]),
            "decision": decision,
        })

recomputed_rows

[{'case_id': '1',
  'split': 'val',
  'frame_id': '002182',
  'risk_indicator': '1',
  'reason_field': 'Vehicle at 6.5m',
  'weather': 'cloudy',
  'time_of_day': 'day',
  'road_condition': 'normal road',
  'context_score_c': '0.281',
  'D': '0.650',
  'r': '0.350',
  'mu_p': '0.701',
  'nu_p': '0.250',
  'pi_p': '0.049',
  'zeta_p': '0.000',
  'M': '0.703',
  'decision': 'decisive braking'},
 {'case_id': '2',
  'split': 'val',
  'frame_id': '000325',
  'risk_indicator': '0',
  'reason_field': 'No risk label',
  'weather': 'rain',
  'time_of_day': 'night',
  'road_condition': 'wet road',
  'context_score_c': '0.815',
  'D': '',
  'r': '',
  'mu_p': '0.344',
  'nu_p': '0.646',
  'pi_p': '0.009',
  'zeta_p': '0.000',
  'M': '0.345',
  'decision': 'cautious monitoring'},
 {'case_id': '3',
  'split': 'test',
  'frame_id': '002044',
  'risk_indicator': '1',
  'reason_field': 'Vehicle at -2.3m',
  'weather': 'rain',
  'time_of_day': 'day',
  'road_condition': 'wet road',
  'context_score_c': 

In [5]:
# Display the recomputed table.
#
# This cell assumes that the previous recomputation cell has been run.
# If it has not, the message below explains what to do instead of raising
# a NameError.

if "recomputed_rows" not in globals():
    raise RuntimeError(
        "The variable 'recomputed_rows' has not been created yet. "
        "Please run the previous cell titled 'Load the selected rows and recompute all "
        "reported quantities', or use Kernel -> Restart & Run All."
    )

try:
    import pandas as pd

    df = pd.DataFrame(recomputed_rows)
    display(df)
except ImportError:
    for row in recomputed_rows:
        print(row)

,case_id,split,frame_id,risk_indicator,reason_field,weather,time_of_day,road_condition,context_score_c,D,r,mu_p,nu_p,pi_p,zeta_p,M,decision
0,1,val,002182,1,Vehicle at 6.5m,cloudy,day,normal road,0.281,0.650,0.350,0.701,0.250,0.049,0.000,0.703,decisive braking
1,2,val,000325,0,No risk label,rain,night,wet road,0.815,,,0.344,0.646,0.009,0.000,0.345,cautious monitoring
2,3,test,002044,1,Vehicle at -2.3m,rain,day,wet road,0.608,0.000,1.000,0.961,0.202,0.000,0.163,0.948,decisive braking


In [6]:
# Save the recomputed results.
#
# This cell also requires the recomputation cell to have been run.

if "recomputed_rows" not in globals():
    raise RuntimeError(
        "The variable 'recomputed_rows' has not been created yet. "
        "Please run the previous recomputation cell first, or use Kernel -> Restart & Run All."
    )

fieldnames = list(recomputed_rows[0].keys())

with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(recomputed_rows)

print(f"Wrote {OUTPUT_FILE.resolve()}")

Wrote /Users/oross/Library/CloudStorage/Dropbox/2026/Articulos_Caps_sometidos_2026/Contradiction_Aware_FSS_2026/Suplementary/1st_Submission/MFL_ZOD_Supplementary_Material/zod_mfl_recomputed_results.csv


## Verification against expected values

The input file contains the manuscript values in `expected_*` columns.
The following cell checks that the recomputed values agree with those values
after rounding to three decimals.

This is a deterministic reproducibility check, not an empirical validation.

In [7]:
# Verify the recomputed values against the expected values.
#
# This cell requires the recomputation cell to have been run.

if "recomputed_rows" not in globals():
    raise RuntimeError(
        "The variable 'recomputed_rows' has not been created yet. "
        "Please run the previous recomputation cell first, or use Kernel -> Restart & Run All."
    )

tolerance = 1.1e-3
verification_rows = []

with INPUT_FILE.open(newline="", encoding="utf-8") as f:
    source_rows = list(csv.DictReader(f))

fields_to_check = [
    ("context_score_c", "context_score_c"),
    ("mu_p", "expected_mu_p"),
    ("nu_p", "expected_nu_p"),
    ("pi_p", "expected_pi_p"),
    ("zeta_p", "expected_zeta_p"),
    ("M", "expected_M"),
]

all_ok = True

for recomputed, source in zip(recomputed_rows, source_rows):
    case_id = recomputed["case_id"]

    for computed_field, expected_field in fields_to_check:
        computed_value = float(recomputed[computed_field])
        expected_value = float(source[expected_field])
        ok = math.isclose(computed_value, expected_value, abs_tol=tolerance)
        all_ok = all_ok and ok

        verification_rows.append({
            "case_id": case_id,
            "quantity": computed_field,
            "computed": f"{computed_value:.3f}",
            "expected": f"{expected_value:.3f}",
            "ok": ok,
        })

    decision_ok = recomputed["decision"] == source["reported_decision"]
    all_ok = all_ok and decision_ok
    verification_rows.append({
        "case_id": case_id,
        "quantity": "decision",
        "computed": recomputed["decision"],
        "expected": source["reported_decision"],
        "ok": decision_ok,
    })

print("All checks passed within the verification tolerance." if all_ok else "At least one check failed.")

try:
    import pandas as pd
    display(pd.DataFrame(verification_rows))
except ImportError:
    for row in verification_rows:
        print(row)

All checks passed within the verification tolerance.


,case_id,quantity,computed,expected,ok
0,1,context_score_c,0.281,0.281,True
1,1,mu_p,0.701,0.701,True
2,1,nu_p,0.250,0.250,True
3,1,pi_p,0.049,0.049,True
4,1,zeta_p,0.000,0.000,True
5,1,M,0.703,0.703,True
6,1,decision,decisive braking,decisive braking,True
7,2,context_score_c,0.815,0.815,True
8,2,mu_p,0.344,0.345,True
9,2,nu_p,0.646,0.646,True


## How to read the verification table

The verification table is an internal reproducibility check. It compares the values
recomputed by the notebook against the values reported in the input CSV or in the
manuscript.

The table is written in **long format**. This means that each selected case appears
in several rows: one row for each quantity being checked.

The columns should be read as follows:

- `case_id`: identifies the illustrative case.
- `quantity`: indicates which value is being verified.
- `computed`: value recomputed by the notebook.
- `expected`: value reported in the input CSV or manuscript.
- `ok`: whether the recomputed value agrees with the expected value within the
  verification tolerance.

For example, the row

| case_id | quantity | computed | expected | ok |
|---|---|---:|---:|---|
| 1 | context_score_c | 0.281 | 0.281 | True |

means that, for **Case 1**, the notebook recomputed the contextual uncertainty
score $c$ as $0.281$, and this agrees with the expected value $0.281$.

The row

| case_id | quantity | computed | expected | ok |
|---|---|---:|---:|---|
| 2 | mu_p | 0.344 | 0.345 | True |

means that, for **Case 2**, the notebook recomputed the truth-support value
$\mu_p$. The displayed values differ by $0.001$ because this number lies on a
three-decimal rounding boundary. The check is still marked as `True` because the
values agree within the verification tolerance.

The row

| case_id | quantity | computed | expected | ok |
|---|---|---|---|---|
| 2 | decision | cautious monitoring | cautious monitoring | True |

means that, for **Case 2**, the notebook recomputed the final safety-first decision
as `cautious monitoring`, and this matches the expected decision.

The main quantities are:

- `context_score_c`: contextual uncertainty score $c$.
- `mu_p`: truth-support value $\mu_p$.
- `nu_p`: falsity-support value $\nu_p$.
- `pi_p`: hesitation degree $\pi_p$.
- `zeta_p`: contradiction degree $\zeta_p$.
- `M`: mediative score $M(\mu_p,\nu_p)$.
- `decision`: final safety-first decision label.

The possible decision labels are:

- `decisive braking`
- `cautious deceleration and additional sensing`
- `cautious monitoring`

This table is not an additional experiment. It only verifies that the notebook
reproduces the deterministic calculations used in the illustrative case study.

## Interpretation of the selected rows

The three rows are not intended to form an empirical benchmark. They were selected
to illustrate how the same deterministic coding rules behave under three different
evidence patterns: incomplete evidence, difficult non-risk context, and explicit
contradiction.

### Case 1: positive risk indication with incomplete evidence

Case 1 corresponds to a validation frame in which the reason field reports a vehicle
at 6.5 m under cloudy daytime conditions and normal road conditions. Since the risk
indicator is positive, the coding rule uses the reported distance to compute the
proximity quantities $D$ and $r$. A closer object gives stronger support for the
proposition $p$, whereas larger distance contributes more support for the absence of
an immediate obstacle.

The recomputed truth--falsity pair is approximately

$$
(\mu_p,\nu_p) = (0.701,0.250).
$$

Since $\mu_p+\nu_p<1$, this row is interpreted as incomplete rather than
contradictory. The hesitation degree is positive,

$$
\pi_p = 0.049,
$$

while the contradiction degree is zero,

$$
\zeta_p = 0.
$$

The resulting mediative score is

$$
M(\mu_p,\nu_p) \approx 0.703.
$$

This value is slightly above the braking threshold $T_{\mathrm{brake}}=0.7$.
Therefore, the row gives a borderline safety-first braking recommendation. The
nonzero hesitation value indicates that the decision is not based on fully complete
evidence, so additional sensing would still be valuable.

### Case 2: difficult non-risk context

Case 2 corresponds to a validation frame with a non-risk label, but the contextual
conditions are difficult: rain, night, wet road, and a dense vulnerable-road-user
scene. Because the risk indicator is zero, the non-risk coding rule assigns stronger
support to the absence of a safety-relevant obstacle. However, the high contextual
uncertainty score still creates residual support for $p$.

The recomputed pair is approximately

$$
(\mu_p,\nu_p) = (0.345,0.646).
$$

Here $\mu_p+\nu_p$ is very close to 1 but still slightly below it, so the row has a
small hesitation degree and no contradiction:

$$
\pi_p = 0.009,\qquad \zeta_p=0.
$$

The mediative score is

$$
M(\mu_p,\nu_p) \approx 0.345.
$$

This score is below the braking and slow-down thresholds. Thus, the rule does not
recommend emergency braking. Nevertheless, the difficult context justifies cautious
monitoring and additional sensing when available. This row illustrates that a
non-risk label does not eliminate contextual uncertainty; it only shifts the
truth--falsity balance toward the absence of an immediate obstacle.

### Case 3: positive risk indication with contradictory evidence

Case 3 corresponds to a test frame in which the reason field reports a vehicle at
$-2.3$ m under rainy wet-road conditions. The negative distance is clipped to $D=0$,
which corresponds to maximal proximity support $r=1$ in this illustrative coding
scheme. Because the risk indicator is positive and the object is treated as extremely
close, support for $p$ becomes high.

The recomputed pair is approximately

$$
(\mu_p,\nu_p) = (0.961,0.202).
$$

In this case,

$$
\mu_p+\nu_p = 1.163 > 1,
$$

so the row is explicitly overdetermined. The hesitation degree vanishes, while the
contradiction degree is positive:

$$
\pi_p=0,\qquad \zeta_p=0.163.
$$

The mediative score is

$$
M(\mu_p,\nu_p) \approx 0.948.
$$

This is a decisive braking case. The contradiction does not make the evaluation
trivial or undefined. Instead, it is represented explicitly by $\zeta_p$ and absorbed
by the mediative operator, which still returns a bounded and interpretable
safety-first score.

## Reproducibility notes

- The constants are fixed a priori for the illustrative computation.
- No perception model is trained or re-estimated.
- No external package is strictly required; `pandas` is used only for nicer display.
- The output CSV can be regenerated by rerunning all cells.